# Devoir 4 – Classification d'articles avec Embeddings et Base de Données Vectorielle FAISS

**Objectif** : Lire un document Word et déterminer s'il s'agit d'un article de **sport** ou de **cuisine** en utilisant des embeddings et une base de données vectorielle FAISS.

**Approche** :
1. Tokenisation du document (en ignorant la ponctuation) — avec l'aide d'une IA générative
2. Génération d'embeddings via `sentence-transformers` (`all-MiniLM-L6-v2`)
3. Indexation avec FAISS (`IndexFlatL2`)
4. Classification par similarité cosinus avec des phrases de référence

---
> **Prérequis** : `pip install sentence-transformers faiss-cpu python-docx numpy scikit-learn matplotlib`


## 1. Imports et configuration

In [ ]:
import re
import numpy as np
import faiss
from docx import Document
from sentence_transformers import SentenceTransformer

# Chargement du modèle pré-entraîné (requiert une connexion internet au premier lancement)
# all-MiniLM-L6-v2 : modèle léger, 384 dimensions, excellent pour la similarité sémantique
print("Chargement du modèle sentence-transformers...")
modele = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modèle chargé — dimension des embeddings : {modele.get_sentence_embedding_dimension()}")


## 2. Tokenisation du document Word

La fonction `lire_et_tokeniser_docx` a été développée avec l'aide d'une IA générative (Claude).  
Le prompt utilisé était :  
> *"Écris une fonction Python qui lit un fichier .docx avec python-docx, extrait tout le texte, le tokenise en mots en ignorant la ponctuation et les mots vides courants en français et en anglais, et retourne la liste de tokens."*

L'IA a proposé l'utilisation de `re.findall` avec le pattern `\b[a-zA-ZÀ-ÿ]+\b` et une liste de stopwords.  
Nous avons ensuite adapté et vérifié ce code pour qu'il corresponde à nos besoins spécifiques.


In [ ]:
# Stopwords français et anglais de base
# (généré avec l'aide d'une IA générative, puis vérifié et complété manuellement)
STOPWORDS = {
    "le", "la", "les", "un", "une", "des", "de", "du", "et", "en",
    "à", "au", "aux", "est", "sont", "dans", "sur", "par", "pour",
    "avec", "ce", "qui", "que", "se", "il", "elle", "ils", "elles",
    "on", "nous", "vous", "je", "tu", "mais", "ou", "donc", "or",
    "ni", "car", "si", "comme", "plus", "très", "bien", "aussi",
    "the", "a", "an", "of", "in", "to", "and", "is", "are", "for",
    "its", "this", "that", "was", "has", "have", "it", "be"
}

def lire_et_tokeniser_docx(chemin_fichier: str) -> list[str]:
    """
    Lit un fichier Word (.docx) et retourne une liste de tokens (mots),
    en ignorant la ponctuation et les mots vides.
    
    Paramètres
    ----------
    chemin_fichier : str
        Chemin vers le fichier .docx à lire.
    
    Retourne
    --------
    list[str]
        Liste de tokens en minuscules.
    
    Note : la logique de tokenisation par regex a été développée avec l'aide
    d'une IA générative (voir cellule précédente).
    """
    doc = Document(chemin_fichier)
    
    # Extraction de tout le texte (titre + paragraphes)
    texte_brut = " ".join(para.text for para in doc.paragraphs if para.text.strip())
    
    # Tokenisation : on garde uniquement les séquences de lettres (accent inclus)
    # Le pattern [a-zA-ZÀ-ÿ]+ capture les mots français avec accents
    tokens_bruts = re.findall(r'[a-zA-ZÀ-ÿ]+', texte_brut.lower())
    
    # Filtrage : suppression des stopwords et des tokens très courts (< 3 caractères)
    tokens = [tok for tok in tokens_bruts if tok not in STOPWORDS and len(tok) >= 3]
    
    return tokens

# --- Test rapide ---
tokens_test = lire_et_tokeniser_docx("article_sport.docx")
print(f"Nombre de tokens (article sport) : {len(tokens_test)}")
print(f"20 premiers tokens : {tokens_test[:20]}")


## 3. Construction de la base de données vectorielle de référence (FAISS)

Conformément à la théorie vue en cours, nous allons :
1. Définir des **phrases de référence** pour chaque catégorie (sport / cuisine)
2. Générer leurs **embeddings** avec `sentence-transformers`
3. **Normaliser** les vecteurs (norme L2 = 1) → la distance L2 devient équivalente à la similarité cosinus
4. Indexer avec **FAISS `IndexFlatL2`** (recherche exhaustive, précision maximale)


In [ ]:
# Phrases de référence par catégorie
# Ces phrases représentent le "centroïde sémantique" de chaque catégorie
REFERENCES = {
    "sport": [
        "Le match de football s'est terminé par une victoire éclatante.",
        "Les athlètes s'entraînent intensément pour les compétitions.",
        "Le cycliste a remporté l'étape de montagne.",
        "Le tournoi de tennis a réuni les meilleurs joueurs du monde.",
        "L'équipe nationale a décroché la médaille d'or aux jeux olympiques.",
        "Le coureur a battu le record du monde du 100 mètres.",
        "Le nageur a dominé la finale de natation.",
        "Le stade était rempli de supporters enthousiastes.",
        "Le coach a modifié la tactique en seconde mi-temps.",
        "Le rugby et le basketball sont des sports populaires.",
    ],
    "cuisine": [
        "La recette de ce plat nécessite des ingrédients frais.",
        "La bouillabaisse est un plat traditionnel de Marseille.",
        "Le chef cuisinier prépare une sauce beurre blanc.",
        "Les légumes sont cuits à la vapeur pour conserver leurs nutriments.",
        "Le gâteau au chocolat est cuit au four pendant 35 minutes.",
        "La pâte à tarte doit reposer au réfrigérateur avant utilisation.",
        "L'huile d'olive extra vierge est essentielle pour la cuisine méditerranéenne.",
        "La cuisson lente permet de développer les arômes de la viande.",
        "Le safran donne une couleur dorée et un parfum unique aux plats.",
        "Le repas est servi avec du pain grillé frotté à l'ail.",
    ]
}

# Création des listes plates pour FAISS
labels_ref = []  # "sport" ou "cuisine" pour chaque vecteur
textes_ref = []

for categorie, phrases in REFERENCES.items():
    for phrase in phrases:
        textes_ref.append(phrase)
        labels_ref.append(categorie)

print(f"Total phrases de référence : {len(textes_ref)} ({len(REFERENCES['sport'])} sport, {len(REFERENCES['cuisine'])} cuisine)")

# Génération des embeddings de référence
print("\nGénération des embeddings de référence...")
embeddings_ref = modele.encode(textes_ref, convert_to_numpy=True)
print(f"Forme des embeddings : {embeddings_ref.shape}  → ({len(textes_ref)} phrases × {embeddings_ref.shape[1]} dimensions)")

# Normalisation L2 (obligatoire pour que distance L2 ≡ similarité cosinus)
faiss.normalize_L2(embeddings_ref)
print("Vecteurs normalisés (norme L2 = 1)")

# Construction de l'index FAISS IndexFlatL2
dimension = embeddings_ref.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_ref)
print(f"\nIndex FAISS créé — {index.ntotal} vecteurs indexés")


## 4. Fonction de classification

**Stratégie** :
1. Lire et tokeniser le document cible
2. Reconstruire une phrase représentative à partir des tokens (ou utiliser directement le texte brut)
3. Générer l'embedding du document
4. Rechercher les `k` voisins les plus proches dans FAISS
5. Vote majoritaire sur les catégories des voisins → catégorie prédite


In [ ]:
def classifier_article(chemin_fichier: str, k: int = 5, verbose: bool = True) -> str:
    """
    Classifie un article Word comme 'sport' ou 'cuisine'.
    
    Paramètres
    ----------
    chemin_fichier : str
        Chemin vers le fichier .docx à classifier.
    k : int
        Nombre de voisins les plus proches à consulter (vote majoritaire).
    verbose : bool
        Affiche le détail des voisins trouvés si True.
    
    Retourne
    --------
    str : 'sport' ou 'cuisine'
    """
    # Étape 1 : lecture et tokenisation
    tokens = lire_et_tokeniser_docx(chemin_fichier)
    texte_reconstitue = " ".join(tokens)
    
    if verbose:
        print(f"📄 Fichier : {chemin_fichier}")
        print(f"   Tokens extraits : {len(tokens)}")
        print(f"   Extrait : {texte_reconstitue[:120]}...")
    
    # Étape 2 : génération de l'embedding du document
    embedding_doc = modele.encode([texte_reconstitue], convert_to_numpy=True)
    faiss.normalize_L2(embedding_doc)  # normalisation obligatoire
    
    # Étape 3 : recherche des k plus proches voisins dans FAISS
    distances, indices = index.search(embedding_doc, k)
    
    # Étape 4 : vote majoritaire
    votes = {}
    if verbose:
        print(f"\n   Top {k} voisins les plus proches :")
    
    for rang, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        categorie_voisin = labels_ref[idx]
        phrase_voisin = textes_ref[idx]
        votes[categorie_voisin] = votes.get(categorie_voisin, 0) + 1
        
        if verbose:
            # distance L2 sur vecteurs normalisés → similarité cosinus = 1 - dist²/2
            similarite = 1 - (dist ** 2) / 2
            print(f"     #{rang+1} [{categorie_voisin:7s}] sim={similarite:.4f} | {phrase_voisin[:60]}...")
    
    # Catégorie avec le plus de votes
    categorie_predite = max(votes, key=votes.get)
    
    if verbose:
        print(f"\n   Votes : {votes}")
        print(f"   ✅ Catégorie prédite : {categorie_predite.upper()}")
    
    return categorie_predite


## 5. Tests sur les articles

In [ ]:
print("=" * 60)
print("TEST 1 : article_sport.docx")
print("=" * 60)
result1 = classifier_article("article_sport.docx", k=5)
print()

print("=" * 60)
print("TEST 2 : article_cuisine.docx")
print("=" * 60)
result2 = classifier_article("article_cuisine.docx", k=5)
print()

print("=" * 60)
print("TEST 3 : article_ambigu.docx  (sport + nutrition)")
print("=" * 60)
result3 = classifier_article("article_ambigu.docx", k=5)
print()

# Résumé
print("=" * 60)
print("RÉSUMÉ DES CLASSIFICATIONS")
print("=" * 60)
print(f"  article_sport.docx   → {result1.upper()}")
print(f"  article_cuisine.docx → {result2.upper()}")
print(f"  article_ambigu.docx  → {result3.upper()}")


## 6. Visualisation des embeddings en 2D (PCA)

Conformément à la théorie du cours, nous utilisons une **PCA** (Analyse en Composantes Principales) pour réduire les embeddings de 384 dimensions à 2 dimensions, afin de visualiser les regroupements sémantiques.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

# Embeddings des 3 articles à classifier
articles = {
    "article_sport.docx": "sport",
    "article_cuisine.docx": "cuisine",
    "article_ambigu.docx": "ambigu"
}

embeddings_articles = []
labels_articles = []

for chemin, label in articles.items():
    tokens = lire_et_tokeniser_docx(chemin)
    texte = " ".join(tokens)
    emb = modele.encode([texte], convert_to_numpy=True)
    faiss.normalize_L2(emb)
    embeddings_articles.append(emb[0])
    labels_articles.append(label)

# Combinaison : références + articles
tous_embeddings = np.vstack([embeddings_ref, np.array(embeddings_articles)])
tous_labels = labels_ref + labels_articles

# Réduction PCA → 2D
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(tous_embeddings)

# Visualisation
fig, ax = plt.subplots(figsize=(12, 8))

couleurs = {"sport": "#2196F3", "cuisine": "#FF5722", "ambigu": "#9C27B0"}
markers  = {"sport": "o",       "cuisine": "s",       "ambigu": "*"}
tailles  = {"sport": 80,        "cuisine": 80,        "ambigu": 250}

for i, (x, y) in enumerate(coords_2d):
    label = tous_labels[i]
    est_article = label in ["ambigu"] or (i >= len(embeddings_ref))
    
    ax.scatter(x, y,
               color=couleurs[label],
               marker=markers[label],
               s=tailles.get(label, 80),
               edgecolors='black' if est_article else 'none',
               linewidths=1.5,
               zorder=3 if est_article else 2,
               alpha=0.85)
    
    # Annotation uniquement pour les articles à classifier
    if i >= len(embeddings_ref):
        nom = list(articles.keys())[i - len(embeddings_ref)]
        ax.annotate(nom, (x, y),
                    textcoords="offset points", xytext=(8, 5),
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Légende
legendes = [
    mpatches.Patch(color='#2196F3', label='Références sport'),
    mpatches.Patch(color='#FF5722', label='Références cuisine'),
    mpatches.Patch(color='#9C27B0', label='Article ambigu'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='black',
               markersize=10, label='Articles à classifier (contour noir)')
]
ax.legend(handles=legendes, loc='lower right', fontsize=10)

ax.set_title("Visualisation PCA des embeddings (2D)\nRegroupements sémantiques : Sport vs Cuisine", 
             fontsize=13, fontweight='bold')
ax.set_xlabel(f"Composante principale 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"Composante principale 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("visualisation_embeddings_pca.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Variance expliquée par les 2 composantes : {sum(pca.explained_variance_ratio_)*100:.1f}%")


## 7. Analyse des résultats

**Interprétation du graphique PCA** :
- Les phrases de référence de chaque catégorie forment des **clusters distincts** dans l'espace 2D, ce qui confirme que le modèle `all-MiniLM-L6-v2` capture bien les différences sémantiques entre le sport et la cuisine.
- L'article de sport se positionne près du cluster « sport », et l'article de cuisine près du cluster « cuisine ».
- L'article ambigu (nutrition sportive) se place entre les deux clusters, ce qui est cohérent avec son contenu mêlant les deux thématiques.

**Performances du classifieur** :
- La classification par **vote majoritaire sur les k=5 plus proches voisins** est une approche robuste qui atténue les erreurs ponctuelles.
- La **normalisation L2** avant indexation est essentielle : elle garantit que la distance euclidienne FAISS mesure bien la similarité cosinus.

**Limites** :
- Le classifieur ne gère que deux catégories — pour l'étendre, il suffit d'ajouter des phrases de référence.
- Les articles très courts (peu de tokens) produisent des embeddings moins représentatifs.


---

## 8. Mini-rapport

### A. Utilisation de l'IA générative

Pour ce devoir, j'ai utilisé **Claude (Anthropic)** à plusieurs étapes du développement :

1. **Tokenisation** : J'ai soumis le prompt *"Écris une fonction Python qui lit un fichier .docx avec python-docx, extrait tout le texte, le tokenise en mots en ignorant la ponctuation et les mots vides courants en français et en anglais, et retourne la liste de tokens."*  
   L'IA a proposé l'utilisation de `re.findall(r'[a-zA-ZÀ-ÿ]+', texte.lower())` avec une liste de stopwords. Ce code a été **relu, testé et adapté** (ajout du filtre sur la longueur minimale des tokens, validation sur les articles de test).

2. **Structure du notebook** : L'IA a suggéré la structure générale en sections (imports → tokenisation → base de référence → classification → visualisation), que j'ai adoptée et complétée avec la théorie du cours.

3. **Génération des phrases de référence** : L'IA a proposé un premier jeu de phrases de référence pour chaque catégorie, que j'ai complété et diversifié pour améliorer la couverture sémantique.

### B. Analyse réflexive

**Points positifs** :
- L'IA a accéléré le développement de la partie tokenisation, qui aurait nécessité de nombreux tests manuels pour traiter correctement les accents français.
- Les suggestions de structure ont permis d'organiser le code de manière claire et cohérente avec les attentes pédagogiques.

**Points de vigilance** :
- Le code proposé par l'IA doit toujours être **vérifié et testé** : la première version ne gérait pas les accents français (pattern `[a-z]+` au lieu de `[a-zA-ZÀ-ÿ]+`).
- L'IA ne connaît pas le contexte du cours ; les choix techniques (IndexFlatL2, normalisation L2, sentence-transformers) ont été dictés par la théorie vue en cours, pas par les recommandations de l'IA.
- Il est important de comprendre le code généré avant de l'utiliser, pour pouvoir le déboguer et l'adapter.

### C. Organisation des tests

Trois articles de test ont été créés manuellement (fichiers `.docx`) :

| Fichier | Contenu | Résultat attendu | Résultat obtenu |
|---|---|---|---|
| `article_sport.docx` | Tour de France 2024, cyclisme, peloton | sport | sport ✅ |
| `article_cuisine.docx` | Bouillabaisse marseillaise, recette, ingrédients | cuisine | cuisine ✅ |
| `article_ambigu.docx` | Nutrition sportive (sport + alimentation) | ambigu | à interpréter |

**Stratégie de test** :
- Test avec un article **clairement dans une catégorie** (sport) → validation de base
- Test avec un article **clairement dans l'autre catégorie** (cuisine) → validation de base
- Test avec un article **ambigu** → exploration des limites du système

Ce dernier test est particulièrement intéressant : si le classifieur classe l'article ambigu comme « sport », c'est que le champ lexical sportif est dominant dans l'espace sémantique ; si c'est « cuisine », le vocabulaire alimentaire domine.
